[![Roboflow Notebooks](https://media.roboflow.com/notebooks/template/bannertest2-2.png?ik-sdk-version=javascript-1.4.3&updatedAt=1672932710194)](https://github.com/roboflow/notebooks)

# How to Track Objects with RF-DETR and McByte Tracker

McByte extends BoT-SORT-style tracking with segmentation masks. It first locks clear IoU matches, then uses masks initialized by Segment Anything (SAM) and propagated by Cutie to resolve ambiguous associations. This extra visual evidence helps preserve identities through occlusion and crowded scenes without training on your video. McByte can also run without masks, but this notebook enables the complete mask-conditioned pipeline.

## Setup

### Check GPU availability

Let's make sure that we have access to a GPU. We can use the `nvidia-smi` command to do that. If no GPU appears, navigate to `Runtime` → `Change runtime type` → `Hardware accelerator`, select a GPU, and click `Save`.

McByte's SAM and Cutie models are compute-intensive, so a GPU runtime is strongly recommended.

In [ ]:
!nvidia-smi

### Install dependencies

The `mask` extra installs the optional SAM and Cutie dependencies used by McByte. The `detection` extra installs the model runner used for RF-DETR inference.

You may see dependency conflict warnings in Google Colab. This is expected for the preinstalled Colab environment and does not affect this notebook.

In [ ]:
!pip install -q "trackers[detection,mask]==2.6.0"

### Download example data

Download two example videos for testing. You can use these or replace them with your own videos.

In [ ]:
!wget -q https://storage.googleapis.com/com-roboflow-marketing/supervision/video-examples/bikes-1280x720-1.mp4
!wget -q https://storage.googleapis.com/com-roboflow-marketing/supervision/video-examples/bikes-1280x720-2.mp4

## Track from CLI

The `trackers` library provides a convenient command-line interface (CLI) for running detection and tracking without writing Python. Here we select McByte, enable its mask manager, and ask SAM and Cutie to run on the GPU.

The first run downloads the RF-DETR, SAM, and Cutie checkpoints, so it takes longer than later runs. For all options, visit the [Track from CLI documentation](https://trackers.roboflow.com/develop/learn/track/).

In [ ]:
SOURCE_VIDEO_PATH = "/content/bikes-1280x720-1.mp4"
TARGET_VIDEO_PATH = "/content/bikes-1280x720-1-mcbyte.mp4"

!trackers track \
    --source {SOURCE_VIDEO_PATH} \
    --output.video {TARGET_VIDEO_PATH} \
    --output.overwrite true \
    --detection.model rfdetr-medium \
    --detection.confidence 0.2 \
    --tracker mcbyte \
    --tracker.enable_mask_manager true \
    --tracker.mask.device cuda \
    --show.trajectories true

In [ ]:
TARGET_VIDEO_COMPRESSED_PATH = "/content/bikes-1280x720-1-mcbyte-compressed.mp4"

!ffmpeg -y -loglevel error -i {TARGET_VIDEO_PATH} -vcodec libx264 -crf 28 {TARGET_VIDEO_COMPRESSED_PATH}

In [ ]:
from IPython.display import Video

Video(TARGET_VIDEO_COMPRESSED_PATH, embed=True, width=1080)

## Track from Python

### Initialize the detector and tracker

`enable_mask_manager=True` turns on McByte's complete SAM + Cutie pipeline. Passing the frame to `tracker.update(...)` is essential: McByte needs it to initialize and propagate masks and to compensate for camera motion.

In [ ]:
from inference_models import AutoModel

from trackers import McByteMaskConfig, McByteTracker

model = AutoModel.from_pretrained("rfdetr-medium", device="cuda")
tracker = McByteTracker(
    enable_mask_manager=True,
    mask_config=McByteMaskConfig(device="cuda"),
)

### Configure annotators

Coloring boxes, labels, and trajectories by track ID makes identity changes easy to spot.

In [ ]:
import supervision as sv

color = sv.ColorPalette.from_hex(
    [
        "#ffff00",
        "#ff9b00",
        "#ff8080",
        "#ff66b2",
        "#ff66ff",
        "#b266ff",
        "#9999ff",
        "#3399ff",
        "#66ffff",
        "#33ff99",
        "#66ff66",
        "#99ff00",
    ]
)

box_annotator = sv.BoxAnnotator(
    color=color,
    color_lookup=sv.ColorLookup.TRACK,
)
label_annotator = sv.LabelAnnotator(
    color=color,
    color_lookup=sv.ColorLookup.TRACK,
    text_color=sv.Color.BLACK,
    text_scale=0.8,
)
trace_annotator = sv.TraceAnnotator(
    color=color,
    color_lookup=sv.ColorLookup.TRACK,
    thickness=2,
    trace_length=100,
)

### Run detection + tracking

McByte receives both the detections and the original frame on every callback. It returns the same `sv.Detections` structure with a stable `tracker_id` assigned to each active track.

In [ ]:
CONFIDENCE_THRESHOLD = 0.2
NMS_THRESHOLD = 0.3

SOURCE_VIDEO_PATH = "/content/bikes-1280x720-2.mp4"
TARGET_VIDEO_PATH = "/content/bikes-1280x720-2-mcbyte.mp4"


def callback(frame, frame_index):
    result = model(frame)[0]
    detections = result.to_supervision()
    detections = detections[detections.confidence >= CONFIDENCE_THRESHOLD]
    detections = detections.with_nms(threshold=NMS_THRESHOLD)
    detections = tracker.update(detections, frame=frame)

    labels = [f"#{tracker_id}" for tracker_id in detections.tracker_id]
    annotated_frame = frame.copy()
    annotated_frame = box_annotator.annotate(annotated_frame, detections)
    annotated_frame = trace_annotator.annotate(annotated_frame, detections)
    annotated_frame = label_annotator.annotate(
        annotated_frame,
        detections,
        labels=labels,
    )
    return annotated_frame


tracker.reset()
sv.process_video(
    source_path=SOURCE_VIDEO_PATH,
    target_path=TARGET_VIDEO_PATH,
    callback=callback,
    show_progress=True,
)

### Display result

In [ ]:
TARGET_VIDEO_COMPRESSED_PATH = "/content/bikes-1280x720-2-mcbyte-compressed.mp4"

!ffmpeg -y -loglevel error -i {TARGET_VIDEO_PATH} -vcodec libx264 -crf 28 {TARGET_VIDEO_COMPRESSED_PATH}

In [ ]:
from IPython.display import Video

Video(TARGET_VIDEO_COMPRESSED_PATH, embed=True, width=1080)

You just combined the McByte tracker with an RF-DETR detector. Nice work!

McByte can also run without SAM and Cutie by constructing `McByteTracker()` with its default settings. That lightweight mode keeps clear-match locking and IoU association, but mask evidence is what gives McByte its distinctive advantage in ambiguous scenes.

Trackers makes it easy to mix and match multi-object tracking algorithms with your favorite detection backends, including Inference, Ultralytics, and Transformers.

Ready to go deeper? Explore McByte's mask conditioning and configuration options in the trackers [Documentation](https://trackers.roboflow.com/develop/trackers/mcbyte/) or dive into the code on [GitHub](https://github.com/roboflow/trackers).

Got feedback or ideas? Open an issue on [GitHub Issues](https://github.com/roboflow/trackers/issues).